# Trimmed turning angle (remove 40 points at the beginning and end) 


In [2]:
import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from matplotlib.colors import Normalize
import plotly.graph_objects as go

# Plotting trimmed turning angle

In [11]:
# ── PATHS ─────────────────────────────────────────────────────────────
TURN_DIR = r"./../../dataFolders/MuscaChasingBeads/Turning_angle/Trimmed_2500fps"
SUB_DIR  = r"./../../dataFolders/MuscaChasingBeads/xyz_Smooth"

OUT_DIR  = r"./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/2500fps_trimmed"
HTML_OUT = os.path.join(OUT_DIR, "ALL_TRIALS_3D_2500fps.html")

os.makedirs(OUT_DIR, exist_ok=True)

FPS = 2500
# TRIM = 40   

# ── GET FILES ─────────────────────────────────────────────────────────
turn_files = glob.glob(os.path.join(TURN_DIR, "*_CENTER.csv"))

if not turn_files:
    raise FileNotFoundError("No turning angle files found")

# ── INTERACTIVE FIGURE ────────────────────────────────────────────────
fig3d = go.Figure()
valid_trials = []
buttons = []

# ── LOOP ─────────────────────────────────────────────────────────────
for path in turn_files:
    fname = os.path.basename(path)

    match = re.search(r"(Trial\d+_\d+rpmxyzpts)", fname)
    if match:
        trial = match.group(1)
    else:
        print(f"  ⚠ Could not extract trial name from {fname}")
        continue
    print(f"\nProcessing: {trial}")

    df = pd.read_csv(path)

    turn_3d = df["turning_angle_deg"].values
    turn_xy = df["turning_angle_xy_deg"].values
    turn_yz = df["turning_angle_yz_deg"].values

    time = np.arange(len(turn_3d)) / FPS

    # ── MATCH TRAJECTORY ─────────────────────────────────────────────
    sub_path = glob.glob(os.path.join(SUB_DIR, f"{trial}*_SMOOTH.csv"))
    if not sub_path:
        print("  ⚠ No trajectory file")
        continue

    df_sub = pd.read_csv(sub_path[0])
    head = df_sub[["pt2_X", "pt2_Y", "pt2_Z"]].values

    # ── ALIGN LENGTHS ────────────────────────────────────────────────
    min_len = min(len(head), len(turn_3d))
    head = head[:min_len]

    turn_3d = turn_3d[:min_len]
    turn_xy = turn_xy[:min_len]
    turn_yz = turn_yz[:min_len]
    time    = time[:min_len]

    x, y, z = head[:, 0], head[:, 1], head[:, 2]

    # ── CLEAN ────────────────────────────────────────────────────────
    valid = np.isfinite(x) & np.isfinite(y) & np.isfinite(z) & \
            np.isfinite(turn_3d) & np.isfinite(turn_xy) & np.isfinite(turn_yz)

    x, y, z = x[valid], y[valid], z[valid]
    turn_3d = turn_3d[valid]
    turn_xy = turn_xy[valid]
    turn_yz = turn_yz[valid]
    time    = time[valid]

    if len(x) < 100:
        print("  ⚠ Too few valid points")
        continue

    # ── TRIM EDGES ───────────────────────────────────────────────────
    x = x[TRIM:-TRIM]
    y = y[TRIM:-TRIM]
    z = z[TRIM:-TRIM]

    turn_3d = turn_3d[TRIM:-TRIM]
    turn_xy = turn_xy[TRIM:-TRIM]
    turn_yz = turn_yz[TRIM:-TRIM]
    time    = time[TRIM:-TRIM]


    # ── 2D PLOTS ─────────────────────────────────────────────────────
    fig2d, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)

    axes[0].plot(time, turn_3d, lw=1)
    axes[0].set_title("Resultant Turning Angle (3D,Smooth)")

    axes[1].plot(time, turn_xy, lw=1)
    axes[1].set_title("Horizontal Turning (XY,Smooth)")

    axes[2].plot(time, turn_yz, lw=1)
    axes[2].set_title("Vertical Turning (YZ,Smooth)")
    axes[2].set_xlabel("Time (s)")

    for ax in axes:
        ax.set_ylabel("deg")
        ax.grid(True, alpha=0.3)

    fig2d.tight_layout()

    fig2d.savefig(os.path.join(OUT_DIR, f"{trial}_2D_turning.png"), dpi=150)
    plt.close(fig2d)

    print("  → Saved 2D plot")

    # ── STATIC 3D ────────────────────────────────────────────────────
    fig_static = plt.figure(figsize=(7,6))
    ax3d = fig_static.add_subplot(111, projection='3d')

    norm = Normalize(vmin=np.min(turn_3d), vmax=np.max(turn_3d))
    cmap = cm.get_cmap("turbo")

    for i in range(len(x)-1):
        ax3d.plot(x[i:i+2], y[i:i+2], z[i:i+2],
                  color=cmap(norm(turn_3d[i])), linewidth=2)

    mappable = cm.ScalarMappable(norm=norm, cmap=cmap)
    fig_static.colorbar(mappable, ax=ax3d, pad=0.1).set_label("Turning Angle (°)")

    ax3d.set_title(f"{trial} — 3D Turning (2500 fps)")
    ax3d.set_xlabel("X")
    ax3d.set_ylabel("Y")
    ax3d.set_zlabel("Z")

    fig_static.savefig(os.path.join(OUT_DIR, f"{trial}_3D_static.png"), dpi=200)
    plt.close(fig_static)

    print("  → Saved static 3D plot")

    # ── INTERACTIVE 3D ───────────────────────────────────────────────
    visible = True if len(valid_trials) == 0 else False

    fig3d.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode="lines+markers",
        visible=visible,
        marker=dict(
            size=3,
            color=turn_3d,
            colorscale="Turbo",
            colorbar=dict(title="Turning Angle (°)")
        ),
        line=dict(
            color=turn_3d,
            colorscale="Turbo",
            width=3
        ),
        name=trial
    ))

    valid_trials.append(trial)

# ── DROPDOWN ─────────────────────────────────────────────────────────
for i, trial in enumerate(valid_trials):
    vis = [False]*len(valid_trials)
    vis[i] = True

    buttons.append(dict(
        label=trial,
        method="update",
        args=[{"visible": vis},
              {"title": trial}]
    ))

fig3d.update_layout(
    updatemenus=[dict(buttons=buttons)],
    title="3D Turning Angle Viewer (2500 fps trimmed)",
    scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
    width=900,
    height=700
)

fig3d.write_html(HTML_OUT)

print("\n DONE: 2500 fps trimmed plots generated")



FileNotFoundError: No turning angle files found

# Saving turning angle csv


In [11]:
# ── PATHS ─────────────────────────────────────────────────────────────
TURN_DIR = r"./../../dataFolders/MuscaChasingBeads/Turning_angle"
SMOOTH_DIR = r"./../../dataFolders/MuscaChasingBeads/xyz_Smooth"

OUT_DIR = r"./../../dataFolders/MuscaChasingBeads/Turning_angle/Trimmed_2500fps"
os.makedirs(OUT_DIR, exist_ok=True)

FPS = 2500
TRIM = 40   # ~15 ms

# ── GET FILES ─────────────────────────────────────────────────────────
turn_files = glob.glob(os.path.join(TURN_DIR, "*_TURNING_ANGLE.csv"))

if not turn_files:
    raise FileNotFoundError("No turning angle files found")

print(f"Found {len(turn_files)} turning files\n")

# ── LOOP ─────────────────────────────────────────────────────────────
for path in turn_files:
    fname = os.path.basename(path)
    base  = os.path.splitext(fname)[0]

    print(f"Processing: {base}")

    # ── EXTRACT TRIAL NAME (robust) ──────────────────────────────────
    match = re.search(r"(Trial\d+_\d+rpmxyzpts)", fname)
    if not match:
        print("  ⚠ Could not extract trial name → skipping")
        continue

    trial = match.group(1)

    # ── LOAD TURNING DATA ────────────────────────────────────────────
    df_turn = pd.read_csv(path)

    if not {"turning_angle_deg", "turning_angle_xy_deg", "turning_angle_yz_deg"}.issubset(df_turn.columns):
        print("  ⚠ Missing turning columns → skipping")
        continue

    turn_3d = df_turn["turning_angle_deg"].values
    turn_xy = df_turn["turning_angle_xy_deg"].values
    turn_yz = df_turn["turning_angle_yz_deg"].values

    time = np.arange(len(turn_3d)) / FPS

    # ── LOAD TRAJECTORY ──────────────────────────────────────────────
    smooth_path = glob.glob(os.path.join(SMOOTH_DIR, f"{trial}*_SMOOTH.csv"))

    if not smooth_path:
        print("  ⚠ No SMOOTH file found → skipping")
        continue

    df_smooth = pd.read_csv(smooth_path[0])

    head = df_smooth[["pt2_X", "pt2_Y", "pt2_Z"]].values

    x, y, z = head[:, 0], head[:, 1], head[:, 2]

    # ── ALIGN LENGTHS ────────────────────────────────────────────────
    min_len = min(len(x), len(turn_3d))
    x, y, z = x[:min_len], y[:min_len], z[:min_len]
    turn_3d = turn_3d[:min_len]
    turn_xy = turn_xy[:min_len]
    turn_yz = turn_yz[:min_len]
    time    = time[:min_len]

    # ── CLEAN NaNs ───────────────────────────────────────────────────
    valid = np.isfinite(x) & np.isfinite(y) & np.isfinite(z) & \
            np.isfinite(turn_3d) & np.isfinite(turn_xy) & np.isfinite(turn_yz)

    x, y, z = x[valid], y[valid], z[valid]
    turn_3d = turn_3d[valid]
    turn_xy = turn_xy[valid]
    turn_yz = turn_yz[valid]
    time    = time[valid]

    if len(x) <= 2 * TRIM:
        print("  ⚠ Too short after cleaning → skipping")
        continue

    # ── TRIM ─────────────────────────────────────────────────────────
    x = x[TRIM:-TRIM]
    y = y[TRIM:-TRIM]
    z = z[TRIM:-TRIM]

    turn_3d = turn_3d[TRIM:-TRIM]
    turn_xy = turn_xy[TRIM:-TRIM]
    turn_yz = turn_yz[TRIM:-TRIM]
    time    = time[TRIM:-TRIM]

    # ── SAVE CLEAN CSV ───────────────────────────────────────────────
    save_name = base.replace("_TURNING_ANGLE", "") + "_TRIMMED_2500fps.csv"
    save_path = os.path.join(OUT_DIR, save_name)

    trimmed_df = pd.DataFrame({
        "time_sec": time,
        "x": x,
        "y": y,
        "z": z,
        "turning_angle_3d_deg": turn_3d,
        "turning_angle_xy_deg": turn_xy,
        "turning_angle_yz_deg": turn_yz,
    })

    trimmed_df.to_csv(save_path, index=False)

    print(f"  → Saved: {save_path}")

print("\n✅ DONE: All trimmed CSVs created")

Found 14 turning files

Processing: Trial2_180rpmxyzpts_CENTER_TURNING_ANGLE
  → Saved: ./../../dataFolders/MuscaChasingBeads/Turning_angle/Trimmed_2500fps\Trial2_180rpmxyzpts_CENTER_TRIMMED_2500fps.csv
Processing: Trial2_180rpmxyzpts_TURNING_ANGLE
  → Saved: ./../../dataFolders/MuscaChasingBeads/Turning_angle/Trimmed_2500fps\Trial2_180rpmxyzpts_TRIMMED_2500fps.csv
Processing: Trial2_200rpmxyzpts_CENTER_TURNING_ANGLE
  → Saved: ./../../dataFolders/MuscaChasingBeads/Turning_angle/Trimmed_2500fps\Trial2_200rpmxyzpts_CENTER_TRIMMED_2500fps.csv
Processing: Trial2_200rpmxyzpts_TURNING_ANGLE
  → Saved: ./../../dataFolders/MuscaChasingBeads/Turning_angle/Trimmed_2500fps\Trial2_200rpmxyzpts_TRIMMED_2500fps.csv
Processing: Trial3_180rpmxyzpts_CENTER_TURNING_ANGLE
  → Saved: ./../../dataFolders/MuscaChasingBeads/Turning_angle/Trimmed_2500fps\Trial3_180rpmxyzpts_CENTER_TRIMMED_2500fps.csv
Processing: Trial3_180rpmxyzpts_TURNING_ANGLE
  → Saved: ./../../dataFolders/MuscaChasingBeads/Turning_angle/

# save smooth trimmed turning angle csvs

In [1]:
import os
import glob
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

# ── PATHS ─────────────────────────────────────────────────────────────
BASE_DIR = r"./../../dataFolders/MuscaChasingBeads/Turning_angle"

INPUT_DIR = os.path.join(BASE_DIR, "Trimmed_2500fps")
OUTPUT_DIR = os.path.join(INPUT_DIR, "Trimmed_2500fps_Smoothed")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── SMOOTHING FUNCTION ───────────────────────────────────────────────
def savgol_smoothening(x, window=81, poly=3):
    pos = np.full_like(x, np.nan)

    isnan = np.isnan(x)
    idx = np.arange(len(x))

    # split into continuous non-NaN segments
    segments = np.split(idx, np.where(isnan)[0])

    for seg in segments:
        if len(seg) < window + 2:
            continue

        xs = x[seg]

        if np.any(~np.isfinite(xs)):
            continue

        if np.nanstd(xs) < 1e-8:
            continue

        pos[seg] = savgol_filter(
            xs,
            window_length=window,
            polyorder=poly,
            mode='interp'
        )

    return pos

# ── LOAD FILES ───────────────────────────────────────────────────────
files = glob.glob(os.path.join(INPUT_DIR, "*_TRIMMED_2500fps.csv"))

print(f"Found {len(files)} trimmed files")

# ── LOOP ─────────────────────────────────────────────────────────────
for path in files:
    fname = os.path.basename(path)
    print(f"\nProcessing: {fname}")

    df = pd.read_csv(path)

    required_cols = [
        "time_sec",
        "turning_angle_3d_deg",
        "turning_angle_xy_deg",
        "turning_angle_yz_deg"
    ]

    if not all(col in df.columns for col in required_cols):
        print("⚠ Missing required columns → skipping")
        continue

    # ── Extract ──────────────────────────────────────────────────────
    time = df["time_sec"].values
    turn_3d = df["turning_angle_3d_deg"].values
    turn_xy = df["turning_angle_xy_deg"].values
    turn_yz = df["turning_angle_yz_deg"].values

    # ── Smooth ───────────────────────────────────────────────────────
    smooth_3d = savgol_smoothening(turn_3d)
    smooth_xy = savgol_smoothening(turn_xy)
    smooth_yz = savgol_smoothening(turn_yz)

    # ── Create output DF ─────────────────────────────────────────────
    out_df = pd.DataFrame({
        "time_sec": time,

        # original
        "turning_angle_3d_deg": turn_3d,
        "turning_angle_xy_deg": turn_xy,
        "turning_angle_yz_deg": turn_yz,

        # smoothed
        "turning_angle_3d_deg_smooth": smooth_3d,
        "turning_angle_xy_deg_smooth": smooth_xy,
        "turning_angle_yz_deg_smooth": smooth_yz,
    })

    # ── Save ─────────────────────────────────────────────────────────
    out_name = fname.replace("_TRIMMED_2500fps.csv", "_SMOOTHED.csv")
    out_path = os.path.join(OUTPUT_DIR, out_name)

    out_df.to_csv(out_path, index=False)

    print(f"  → Saved: {out_path}")

print("\n✅ Done. Smoothed CSVs saved.")

Found 14 trimmed files

Processing: Trial2_180rpmxyzpts_CENTER_TRIMMED_2500fps.csv
  → Saved: ./../../dataFolders/MuscaChasingBeads/Turning_angle\Trimmed_2500fps\Trimmed_2500fps_Smoothed\Trial2_180rpmxyzpts_CENTER_SMOOTHED.csv

Processing: Trial2_180rpmxyzpts_TRIMMED_2500fps.csv
  → Saved: ./../../dataFolders/MuscaChasingBeads/Turning_angle\Trimmed_2500fps\Trimmed_2500fps_Smoothed\Trial2_180rpmxyzpts_SMOOTHED.csv

Processing: Trial2_200rpmxyzpts_CENTER_TRIMMED_2500fps.csv
  → Saved: ./../../dataFolders/MuscaChasingBeads/Turning_angle\Trimmed_2500fps\Trimmed_2500fps_Smoothed\Trial2_200rpmxyzpts_CENTER_SMOOTHED.csv

Processing: Trial2_200rpmxyzpts_TRIMMED_2500fps.csv
  → Saved: ./../../dataFolders/MuscaChasingBeads/Turning_angle\Trimmed_2500fps\Trimmed_2500fps_Smoothed\Trial2_200rpmxyzpts_SMOOTHED.csv

Processing: Trial3_180rpmxyzpts_CENTER_TRIMMED_2500fps.csv
  → Saved: ./../../dataFolders/MuscaChasingBeads/Turning_angle\Trimmed_2500fps\Trimmed_2500fps_Smoothed\Trial3_180rpmxyzpts_CENTE

In [2]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── PATHS ─────────────────────────────────────────────────────────────
INPUT_DIR = r"./../../dataFolders/MuscaChasingBeads/Turning_angle/Trimmed_2500fps/Trimmed_2500fps_Smoothed"

FIG_DIR = r"./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/2500fps_trimmed/smooth"
os.makedirs(FIG_DIR, exist_ok=True)

# ── LOAD FILES ───────────────────────────────────────────────────────
files = glob.glob(os.path.join(INPUT_DIR, "*_SMOOTHED.csv"))

print(f"Found {len(files)} smoothed CSV files")

# ── LOOP ─────────────────────────────────────────────────────────────
for path in files:
    fname = os.path.basename(path)
    print(f"\nProcessing: {fname}")

    df = pd.read_csv(path)

    required_cols = [
        "time_sec",
        "turning_angle_3d_deg", "turning_angle_3d_deg_smooth",
        "turning_angle_xy_deg", "turning_angle_xy_deg_smooth",
        "turning_angle_yz_deg", "turning_angle_yz_deg_smooth"
    ]

    if not all(col in df.columns for col in required_cols):
        print("⚠ Missing required columns → skipping")
        continue

    t = df["time_sec"].values

    # raw
    turn_3d = df["turning_angle_3d_deg"].values
    turn_xy = df["turning_angle_xy_deg"].values
    turn_yz = df["turning_angle_yz_deg"].values

    # smoothed
    smooth_3d = df["turning_angle_3d_deg_smooth"].values
    smooth_xy = df["turning_angle_xy_deg_smooth"].values
    smooth_yz = df["turning_angle_yz_deg_smooth"].values

    # ── PLOT ────────────────────────────────────────────────────────
    fig, axs = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

    # 3D
    axs[0].plot(t, turn_3d, alpha=0.4, label="raw")
    axs[0].plot(t, smooth_3d, label="smoothed")
    axs[0].set_ylabel("3D angle (deg)")
    axs[0].legend()

    # XY
    axs[1].plot(t, turn_xy, alpha=0.4, label="raw")
    axs[1].plot(t, smooth_xy, label="smoothed")
    axs[1].set_ylabel("XY angle (deg)")
    axs[1].legend()

    # YZ
    axs[2].plot(t, turn_yz, alpha=0.4, label="raw")
    axs[2].plot(t, smooth_yz, label="smoothed")
    axs[2].set_ylabel("YZ angle (deg)")
    axs[2].set_xlabel("Time (s)")
    axs[2].legend()

    fig.suptitle(f"Turning Angle (Raw vs Smoothed) — {fname}")
    plt.tight_layout()

    # ── SAVE ────────────────────────────────────────────────────────
    save_name = fname.replace("_SMOOTHED.csv", "_smooth_plot.png")
    save_path = os.path.join(FIG_DIR, save_name)

    plt.savefig(save_path, dpi=300)
    print(f"  → Saved: {save_path}")

    plt.close(fig)

print("\n✅ Done.")

Found 14 smoothed CSV files

Processing: Trial2_180rpmxyzpts_CENTER_SMOOTHED.csv
  → Saved: ./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/2500fps_trimmed/smooth\Trial2_180rpmxyzpts_CENTER_smooth_plot.png

Processing: Trial2_180rpmxyzpts_SMOOTHED.csv
  → Saved: ./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/2500fps_trimmed/smooth\Trial2_180rpmxyzpts_smooth_plot.png

Processing: Trial2_200rpmxyzpts_CENTER_SMOOTHED.csv
  → Saved: ./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/2500fps_trimmed/smooth\Trial2_200rpmxyzpts_CENTER_smooth_plot.png

Processing: Trial2_200rpmxyzpts_SMOOTHED.csv
  → Saved: ./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/2500fps_trimmed/smooth\Trial2_200rpmxyzpts_smooth_plot.png

Processing: Trial3_180rpmxyzpts_CENTER_SMOOTHED.csv
  → Saved: ./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/2500fps_trimmed/smooth\Trial3_180rpmxyzpts_CENTER_smooth_plot.png

Processing: Trial3_180rpmxyzp

In [3]:
import os
import glob
import re
import pandas as pd
import plotly.graph_objects as go

# ── PATHS ─────────────────────────────────────────────────────────────
TURN_DIR   = r"./../../dataFolders/MuscaChasingBeads/Turning_angle/Trimmed_2500fps/Trimmed_2500fps_Smoothed"
SMOOTH_DIR = r"./../../dataFolders/MuscaChasingBeads/xyz_Smooth"

FIG_BASE_DIR = r"./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/2500fps_trimmed/smooth"
os.makedirs(FIG_BASE_DIR, exist_ok=True)

# ==========================================================
# LOOP
# ==========================================================
csv_files = sorted(glob.glob(os.path.join(TURN_DIR, "*_SMOOTHED.csv")))

print(f"\nFound {len(csv_files)} smoothed turning files")

for path in csv_files:
    base = os.path.splitext(os.path.basename(path))[0]

    # ── Extract trial name ───────────────────────────────────────────
    match = re.search(r"(Trial\d+_\d+rpm)", base)
    trial_name = match.group(1) if match else "Trial_Data"

    # ── Save directory (same style as yours) ─────────────────────────
    trial_save_dir = os.path.join(FIG_BASE_DIR, trial_name)
    os.makedirs(trial_save_dir, exist_ok=True)

    print(f"\nProcessing: {trial_name}")

    df = pd.read_csv(path)

    if "turning_angle_3d_deg_smooth" not in df.columns:
        print("  ⚠ Missing smoothed turning column → skipping")
        continue

    turn_3d = df["turning_angle_3d_deg_smooth"].values

    # ── Get corresponding SMOOTH trajectory file ─────────────────────
    smooth_path = glob.glob(os.path.join(SMOOTH_DIR, f"{trial_name}*_SMOOTH.csv"))

    if not smooth_path:
        print(f"  ⚠ No SMOOTH trajectory file found → skipping")
        continue

    df_smooth = pd.read_csv(smooth_path[0])

    head = df_smooth[["pt2_X", "pt2_Y", "pt2_Z"]].values

    # ── Align lengths (IMPORTANT)
    n = min(len(head), len(turn_3d))
    head = head[:n]
    turn_3d = turn_3d[:n]

    x = head[:, 0]
    y = head[:, 1]
    z = head[:, 2]

    # ==========================================================
    # 🔹 INTERACTIVE 3D TRAJECTORY (HEAD-BASED)
    # ==========================================================
    fig = go.Figure(data=go.Scatter3d(
        x=x,
        y=y,
        z=z,
        mode="lines+markers",
        marker=dict(
            size=3,
            color=turn_3d,
            colorscale="Turbo",
            colorbar=dict(title="Turning Angle (°)"),
            showscale=True,
        ),
        line=dict(color=turn_3d, colorscale="Turbo", width=3),
        hovertemplate=(
            "X: %{x:.4f}<br>"
            "Y: %{y:.4f}<br>"
            "Z: %{z:.4f}<br>"
            "Turning Angle: %{marker.color:.2f}°"
            "<extra></extra>"
        ),
    ))

    fig.update_layout(
        title=f"{trial_name} — 3D Head Trajectory (Smoothed Turning)",
        scene=dict(
            xaxis_title="X",
            yaxis_title="Y",
            zaxis_title="Z"
        ),
        width=1000,
        height=700,
    )

    # ── SAVE (same structure style)
    out_html = os.path.join(trial_save_dir, f"{base}_3D_head_trajectory.html")
    fig.write_html(out_html)

    print(f"  → Saved 3D plot: {out_html}")

print(f"\n All plots saved in: {FIG_BASE_DIR}")


Found 14 smoothed turning files

Processing: Trial2_180rpm
  → Saved 3D plot: ./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/2500fps_trimmed/smooth\Trial2_180rpm\Trial2_180rpmxyzpts_CENTER_SMOOTHED_3D_head_trajectory.html

Processing: Trial2_180rpm
  → Saved 3D plot: ./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/2500fps_trimmed/smooth\Trial2_180rpm\Trial2_180rpmxyzpts_SMOOTHED_3D_head_trajectory.html

Processing: Trial2_200rpm
  → Saved 3D plot: ./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/2500fps_trimmed/smooth\Trial2_200rpm\Trial2_200rpmxyzpts_CENTER_SMOOTHED_3D_head_trajectory.html

Processing: Trial2_200rpm
  → Saved 3D plot: ./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/2500fps_trimmed/smooth\Trial2_200rpm\Trial2_200rpmxyzpts_SMOOTHED_3D_head_trajectory.html

Processing: Trial3_180rpm
  → Saved 3D plot: ./../../dataFolders/MuscaChasingBeads/Figures/TurningAnglePlots/2500fps_trimmed/smooth\Trial3_180rpm\Trial3